# Apertus v1.5 zero-shot benchmark

Below is benchmarking the two Apertus models via the Keera API, on the image classification task given by Galaxus.

Both models label the same sample of train images without any finetuning. The answers are scored with the leaderboard
metric against the human labels. The sample comes from a fixed holdout of whole products, so a finetuned model can be
scored on the same images later.

The notebook needs `LLM_BASE_URL` and `LLM_API_KEY` in a `.env` file and the competition data in a `glx_data` folder
next to the repository.

## 1. Config

Every knob is in this cell. `N_RUN` limits how many images of the sample are sent. Answers are stored per image, so a
rerun only sends what is missing.

In [1]:
import base64
import io
import json
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
import requests
from dotenv import find_dotenv, load_dotenv
from PIL import Image

load_dotenv(find_dotenv(usecwd=True))
BASE_URL = os.environ["LLM_BASE_URL"].rstrip("/")
API_KEY = os.environ["LLM_API_KEY"]

MODELS = ["apertus-v1-5-8b", "apertus-v1-5-70b"]
RUN_NAME = "zero_shot_v1"  # rename when the prompt or the image settings change, so runs never mix
N_SAMPLE = 500  # images in the benchmark sample
N_RUN = 500  # how many of them to send now
SEED = 0
IMAGE_SIZE = 512  # images are scaled to about IMAGE_SIZE x IMAGE_SIZE pixels in area
JPEG_QUALITY = 90
WORKERS = 8  # parallel requests; the gateway is shared with the other teams
MAX_TOKENS = 120
MAX_ATTEMPTS = 4
TIMEOUT_SECONDS = 120


def find_upwards(name):
    """The first folder called name in the working directory or one of its parents."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / name).is_dir():
            return folder / name
    raise FileNotFoundError(f"no {name} folder in {Path.cwd()} or above")


DATA_DIR = find_upwards("glx_data")
OUT_DIR = DATA_DIR.parent / "runs"  # outside the repository: the answers belong to competition images
RUN_DIR = OUT_DIR / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(DATA_DIR))
import metric  # the leaderboard metric, from the competition data

DIMENSIONS = metric.DIMENSIONS

## 2. Data

The holdout is 20% of the train products, drawn as in `challenge_description.md`. The benchmark sample is `N_SAMPLE`
images of it. Both id lists are written to disk on the first run and read from there afterwards, so every later run and
the finetuning use exactly the same split.

In [2]:
train = pd.read_csv(DATA_DIR / "train.csv")


def stored_ids(path, make):
    """The ids in path, one per line. The first call creates the file with make()."""
    if not path.exists():
        path.write_text("\n".join(make()) + "\n")
    return path.read_text().split()


holdout_products = stored_ids(
    OUT_DIR / "holdout_products.txt",
    lambda: sorted(train["product_id"].drop_duplicates().sample(frac=0.2, random_state=SEED)),
)
holdout = train[train["product_id"].isin(holdout_products)]
sample_ids = stored_ids(
    OUT_DIR / f"sample_{N_SAMPLE}.txt",
    lambda: list(holdout["image_id"].sample(n=N_SAMPLE, random_state=SEED)),
)
sample = holdout.set_index("image_id").loc[sample_ids].reset_index()
print(f"train {len(train)} images | holdout {len(holdout)} images of {len(holdout_products)} products | sample {len(sample)}")

train 11876 images | holdout 2444 images of 1452 products | sample 500


## 3. Scoring

Only human labels in relevant cells are scored, as on the leaderboard. `solution_frame` empties every other cell.

Two checks before any model runs. The true labels must score exactly 1.0. The constant answer of
`sample_submission.csv` gives the floor that any model has to beat.

In [3]:
CONSTANT_ANSWER = {
    "subject": "full_product",
    "background": "white",
    "perspective": "front_head_on",
    "individuals": "none",
    "text_classification": "none",
    "text_language": "none",
}


def solution_frame(rows):
    """The labels that count: human labels in relevant cells, every other cell empty."""
    solution = rows[["image_id", *DIMENSIONS]].copy()
    for dimension in DIMENSIONS:
        scored = (rows[f"{dimension}_source"] == "human") & rows[f"{dimension}_relevant"]
        solution[dimension] = solution[dimension].where(scored)
    return solution


def constant_frame(rows):
    """The constant answer for every image."""
    return rows[["image_id"]].assign(**CONSTANT_ANSWER)


assert metric.score(solution_frame(sample), sample[["image_id", *DIMENSIONS]], "image_id") == 1.0
print("scored cells:", solution_frame(sample)[list(DIMENSIONS)].notna().sum().to_dict())
print(f"constant answer: {metric.score(solution_frame(sample), constant_frame(sample), 'image_id'):.3f}")

scored cells: {'subject': 474, 'background': 413, 'perspective': 328, 'individuals': 413, 'text_classification': 459, 'text_language': 210}
constant answer: 0.188


## 4. Images

Transparency goes onto white, as the shop and the labelers saw it. Then the image is scaled to about `IMAGE_SIZE`
squared pixels with its aspect ratio kept. Apertus spends roughly one token per 16 x 16 pixels, so the size sets the
cost of a request.

In [4]:
def opaque_rgb_on_white(image):
    """RGB, with any transparency composited onto white (the function of the competition README)."""
    if image.mode not in ("RGBA", "LA", "PA") and "transparency" not in image.info:
        return image.convert("RGB")
    rgba = image.convert("RGBA")
    canvas = Image.new("RGBA", rgba.size, (255, 255, 255, 255))
    canvas.alpha_composite(rgba)
    return canvas.convert("RGB")


def prepared_image(image_path):
    """The image as the model gets it: on white, about IMAGE_SIZE squared pixels, aspect ratio kept."""
    with Image.open(image_path) as image:
        rgb = opaque_rgb_on_white(image)
    scale = IMAGE_SIZE / (rgb.width * rgb.height) ** 0.5
    scale = min(scale, 4 * IMAGE_SIZE / max(rgb.size))  # a thin strip would get a very long side otherwise
    size = (max(32, round(rgb.width * scale)), max(32, round(rgb.height * scale)))
    return rgb.resize(size, Image.LANCZOS)


def data_url(image):
    """A PIL image as a base64 JPEG data URL."""
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=JPEG_QUALITY)
    return "data:image/jpeg;base64," + base64.b64encode(buffer.getvalue()).decode()


example = sample.iloc[0]
print("prepared size of the first sample image:", prepared_image(DATA_DIR / example["image_path"]).size)

prepared size of the first sample image: (442, 592)


## 5. Prompt

A zero-shot model has no labels to learn the conventions from, so the system prompt carries a short version of
`labeling_guideline.md`. It is the same for every image, which lets the server cache it. The answer is held to a JSON
schema, so every class name is valid.

In [5]:
CLASSES = {  # in the order of the labeling guideline
    "subject": [
        "full_product", "detail_closeup", "accessories", "packaging", "package_and_product", "multipack_trade_unit",
        "delivery_scope", "product_grouping", "multiview", "energy_label", "infographic", "logo", "placeholder",
    ],
    "background": ["white", "color", "scene_lifestyle"],
    "perspective": [
        "front_head_on", "front_angle_left", "front_angle_right", "back_head_on", "back_angle_left",
        "back_angle_right", "left_side", "right_side", "top_down", "bottom_up",
    ],
    "individuals": ["individual_present", "none"],
    "text_classification": [
        "nutritional_info", "technical_specs", "promotional_text", "certification_seal", "other", "none",
    ],
    "text_language": ["German", "English", "French", "Italian", "Dutch", "other", "none"],
}
assert all(set(CLASSES[dimension]) == metric.VALID_CLASSES[dimension] for dimension in DIMENSIONS)

SYSTEM_PROMPT = """You label one product image of an online shop along six dimensions. The product context under the image says what is being sold. Judge only what is visible. Answer with one JSON object that holds exactly one class per dimension.

subject: what the image shows
- full_product: the item that is sold, whole, nothing cut off. Also with dimension lines or marketing text added on top, in a lifestyle scene, a book cover, or food, drinks, pet food and supplements in their own pack, front or back.
- detail_closeup: a zoomed part or texture, or the product cut off at the frame. Also the inner pages of a book.
- accessories: an accessory or peripheral, not the main product.
- packaging: an outer box or carton with no product visible. Food in its own pack is full_product.
- package_and_product: product and packaging both visible, also through a window or a blister.
- multipack_trade_unit: several identical units of the same size in the same orientation.
- delivery_scope: the main product with what comes with it, such as accessories, parts and cables.
- product_grouping: several variants of the same product, such as colors or sizes.
- multiview: one product shown two or more times from different angles or in different states.
- energy_label: the EU energy efficiency label.
- infographic: no product photo, for example a text page, table, chart, diagram, icon collage, line drawing or a certification mark alone.
- logo: one brand logo and nothing else.
- placeholder: a generic "no image available" picture.

background: the physical backdrop
- white: pure white, no backdrop at all (a cutout), or content that fills the whole frame.
- color: a uniform backdrop that is not pure white, also light grey, cream, black or a smooth gradient.
- scene_lifestyle: any environment, surface or props. If unsure between color and scene_lifestyle, choose scene_lifestyle.

perspective: find the front face of the product, then judge where it points. Think of a clock face seen from above with the camera at 6 o'clock.
- front_head_on: the front faces the camera.
- front_angle_left: the front points to the lower left of the image (7 to 8 o'clock).
- front_angle_right: the front points to the lower right of the image (4 to 5 o'clock).
- left_side: a pure profile, the front points to the left edge of the image.
- right_side: a pure profile, the front points to the right edge of the image.
- back_head_on: the back faces the camera.
- back_angle_left: the back is toward the camera and the front points to the upper right (1 to 2 o'clock).
- back_angle_right: the back is toward the camera and the front points to the upper left (10 to 11 o'clock).
- top_down: the camera looks straight down at the top surface.
- bottom_up: the camera looks straight up at the underside.
Front faces: for screens the screen. For mice, keyboards and remotes the top surface, with the tip or cable end as the front edge. For kettles, mugs and pans the spout is the front and the handle the back. For shoes the toe is the front edge and a sole toward the camera is bottom_up. For clothing the chest.

individuals
- individual_present: a real person, a real hand or a real animal is in the photo.
- none: otherwise. People printed on a pack, shown on a screen, on a cover or drawn do not count.

text_classification: only elements added digitally on top of the photo, plus nutrition tables. Text printed on the product or its pack is none.
- nutritional_info: a nutrition table is visible, also when printed on the pack. It wins over every other class.
- technical_specs: dimension lines, measurements, size tags or callouts pointing to parts.
- promotional_text: marketing claims, slogans, sale banners or NEW badges.
- certification_seal: trust marks, eco-labels or award badges.
- other: icons, pictograms, stickers or watermarks.
- none: nothing was added.

text_language: the dominant language of the words that say something, printed or added. Brand and model names, technical terms such as USB-C or LED, numbers and units do not count.
- German, English, French, Italian, Dutch: that language covers the most area.
- other: any other language.
- none: no words that count."""

RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {dimension: {"type": "string", "enum": CLASSES[dimension]} for dimension in DIMENSIONS},
    "required": list(DIMENSIONS),
    "additionalProperties": False,
}


def user_prompt(row):
    """The product context of an image."""
    return (
        f"Product: {row['product_title']}\nBrand: {row['brand']}\nProduct type: {row['product_type']}\n"
        f"Category: {row['category']} / {row['subcategory']}"
    )


print(f"system prompt: {len(SYSTEM_PROMPT.split())} words")

system prompt: 706 words


## 6. Client and parser

`classify` sends one image and returns the raw answer with its latency and token counts. A failed request returns the
error instead of raising, so one bad image never stops a run. `parse_answer` turns the raw answer into six class names.
A class it cannot read stays empty and counts as wrong.

The last lines send one image to each model as a smoke test.

In [6]:
RETRY_STATUS = {429, 500, 502, 503, 504}
session = requests.Session()
session.headers["Authorization"] = f"Bearer {API_KEY}"
session.mount("https://", requests.adapters.HTTPAdapter(pool_maxsize=WORKERS))


def post_chat(body):
    """A chat completion. Rate limits, server errors and network errors are retried with a growing pause."""
    for attempt in range(MAX_ATTEMPTS):
        try:
            response = session.post(f"{BASE_URL}/chat/completions", json=body, timeout=TIMEOUT_SECONDS)
            if response.status_code not in RETRY_STATUS:
                response.raise_for_status()
                return response.json()
            problem = f"HTTP {response.status_code}"
        except (requests.ConnectionError, requests.Timeout) as error:
            problem = type(error).__name__
        time.sleep(2**attempt)
    raise RuntimeError(f"gave up after {MAX_ATTEMPTS} attempts, last problem: {problem}")


def classify(row, model):
    """The raw answer of a model for one image, with latency and token counts, or the error."""
    record = {"image_id": row["image_id"], "model": model, "answer": None, "error": None}
    try:
        image = prepared_image(DATA_DIR / row["image_path"])
        body = {
            "model": model,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": [
                        {"type": "image_url", "image_url": {"url": data_url(image)}},
                        {"type": "text", "text": user_prompt(row)},
                    ],
                },
            ],
            "temperature": 0,
            "max_tokens": MAX_TOKENS,
            "response_format": {
                "type": "json_schema",
                "json_schema": {"name": "image_labels", "schema": RESPONSE_SCHEMA, "strict": True},
            },
        }
        started = time.perf_counter()
        reply = post_chat(body)
        record["latency_s"] = round(time.perf_counter() - started, 3)
        record["answer"] = reply["choices"][0]["message"]["content"]
        record["prompt_tokens"] = reply["usage"]["prompt_tokens"]
        record["completion_tokens"] = reply["usage"]["completion_tokens"]
    except Exception as error:  # any failure is reported per image instead of stopping the run
        record["error"] = f"{type(error).__name__}: {error}"[:300]
    return record


CANONICAL = {dimension: {name.lower(): name for name in CLASSES[dimension]} for dimension in DIMENSIONS}


def parse_answer(answer):
    """The class per dimension of a raw answer. A missing or unknown class stays None."""
    try:
        parsed = json.loads(answer[answer.find("{") : answer.rfind("}") + 1])
    except (ValueError, AttributeError):
        parsed = {}
    if not isinstance(parsed, dict):
        parsed = {}
    labels = {}
    for dimension in DIMENSIONS:
        value = parsed.get(dimension)
        labels[dimension] = CANONICAL[dimension].get(value.strip().lower()) if isinstance(value, str) else None
    return labels


checked = parse_answer('{"subject": "Full_Product", "background": "blue"}')
assert checked["subject"] == "full_product", "class names are matched without regard to case"
assert checked["background"] is None, "an unknown class stays empty"
assert checked["individuals"] is None, "a missing dimension stays empty and does not turn into the class none"
assert parse_answer("no json here") == dict.fromkeys(DIMENSIONS)

for model in MODELS:
    record = classify(example, model)
    readable = sum(value is not None for value in parse_answer(record["answer"]).values())
    print(f"{model}: {record['error'] or 'ok'}, {readable} of {len(DIMENSIONS)} classes readable, {record.get('latency_s')} s")

apertus-v1-5-8b: ok, 6 of 6 classes readable, 0.821 s
apertus-v1-5-70b: ok, 6 of 6 classes readable, 1.16 s


## 7. Run

Every answer is appended to `RUN_DIR/<model>.jsonl` as soon as it arrives. Images that already have an answer are
skipped, so the cell can be stopped and rerun at any time. Failed requests are not stored and are sent again on the next
run.

In [7]:
def run_model(model, rows):
    """Send every image of rows that has no stored answer yet. Returns the images per second of this call."""
    path = RUN_DIR / f"{model}.jsonl"
    done = {json.loads(line)["image_id"] for line in path.read_text().splitlines()} if path.exists() else set()
    todo = [row for row in rows.to_dict("records") if row["image_id"] not in done]
    print(f"{model}: {len(rows) - len(todo)} of {len(rows)} already answered, sending {len(todo)}")
    if not todo:
        return None
    failures = []
    started = time.perf_counter()
    with path.open("a") as answers, ThreadPoolExecutor(max_workers=WORKERS) as pool:
        for count, record in enumerate(pool.map(lambda row: classify(row, model), todo), start=1):
            if record["error"]:
                failures.append(record)
            else:
                answers.write(json.dumps(record) + "\n")
                answers.flush()
            if count % 50 == 0 or count == len(todo):
                print(f"  {count} / {len(todo)}, {count / (time.perf_counter() - started):.1f} images/s")
    if failures:
        print(f"  {len(failures)} failed, first error: {failures[0]['error']}")
    return round(len(todo) / (time.perf_counter() - started), 2)


throughput = {model: run_model(model, sample.head(N_RUN)) for model in MODELS}

apertus-v1-5-8b: 500 of 500 already answered, sending 0
apertus-v1-5-70b: 500 of 500 already answered, sending 0


## 8. Results

Both models are scored on the images that each of them has answered, so the comparison is fair even when one run is
incomplete. The constant answer is the floor.

Macro F1 is the leaderboard number. Weighted F1 follows the frequent classes, which are most of the images in the shop.
On a small sample the rare classes hold only a few cells each, so read the macro scores with care.

In [8]:
def read_predictions(model):
    """The stored answers of a model, one row per image, with the parsed class per dimension."""
    path = RUN_DIR / f"{model}.jsonl"
    records = [json.loads(line) for line in path.read_text().splitlines()] if path.exists() else []
    columns = ["image_id", "answer", "latency_s", "prompt_tokens", "completion_tokens"]
    frame = pd.DataFrame(records, columns=columns).drop_duplicates("image_id", keep="last").reset_index(drop=True)
    parsed = pd.DataFrame([parse_answer(answer) for answer in frame["answer"]], columns=list(DIMENSIONS))
    return pd.concat([frame, parsed], axis=1)


predictions = {model: read_predictions(model) for model in MODELS}
common_ids = set(sample["image_id"]).intersection(*(set(frame["image_id"]) for frame in predictions.values()))
assert common_ids, "no image has an answer from every model yet: run section 7 first"

scored_sample = sample[sample["image_id"].isin(common_ids)]
solution = solution_frame(scored_sample)
submissions = {
    model: frame.loc[frame["image_id"].isin(common_ids), ["image_id", *DIMENSIONS]]
    for model, frame in predictions.items()
}
submissions["constant answer"] = constant_frame(scored_sample)
metrics = {name: metric.evaluate(solution, submission, "image_id") for name, submission in submissions.items()}

score_rows = [*(dimension for dimension in DIMENSIONS if dimension in metrics[MODELS[0]]), "overall"]
score_table = pd.DataFrame(
    {
        f"{name} {kind}": [metrics[name][row][f"f1_{kind}"] for row in score_rows]
        for name in metrics
        for kind in ("macro", "weighted")
    },
    index=score_rows,
)
score_table.insert(0, "scored cells", [metrics[MODELS[0]][row]["support"] for row in score_rows])
print(f"{len(common_ids)} images answered by every model")
score_table.round(3)

500 images answered by every model


,scored cells,apertus-v1-5-8b macro,apertus-v1-5-8b weighted,apertus-v1-5-70b macro,apertus-v1-5-70b weighted,constant answer macro,constant answer weighted
subject,474,0.058,0.452,0.058,0.451,0.058,0.452
background,413,0.545,0.786,0.475,0.830,0.308,0.791
perspective,328,0.070,0.159,0.056,0.156,0.047,0.142
individuals,413,0.546,0.944,0.785,0.971,0.489,0.939
text_classification,459,0.150,0.752,0.272,0.767,0.152,0.759
text_language,210,0.352,0.619,0.222,0.424,0.072,0.171
overall,2297,0.287,0.634,0.311,0.631,0.188,0.592


In [9]:
speed_table = pd.DataFrame(
    {
        model: {
            "images answered": len(frame),
            "unreadable classes": int(frame[list(DIMENSIONS)].isna().sum().sum()),
            "median latency in s": frame["latency_s"].median(),
            "mean prompt tokens": round(frame["prompt_tokens"].mean()),
            "mean answer tokens": round(frame["completion_tokens"].mean()),
            f"images/s with {WORKERS} workers, last run": throughput.get(model),
        }
        for model, frame in predictions.items()
    }
)
speed_table

,apertus-v1-5-8b,apertus-v1-5-70b
images answered,500.000,500.0000
unreadable classes,0.000,0.0000
median latency in s,1.253,4.4225
mean prompt tokens,2078.000,2078.0000
mean answer tokens,46.000,53.0000
"images/s with 8 workers, last run",NaN,NaN


## 9. Errors

Two views of where the models fail. The F1 per class shows the weak classes. The confusion matrix shows what a class is
mistaken for, which matters most for perspective, where zero-shot models tend to mix up left and right.

In [10]:
def class_table(dimension):
    """F1 per class of a dimension for every model, with the scored cells per class."""
    per_class = {model: metrics[model][dimension]["per_class"] for model in MODELS}
    classes = [name for name in CLASSES[dimension] if any(name in values for values in per_class.values())]
    table = pd.DataFrame(
        {model: [values.get(name, {}).get("f1") for name in classes] for model, values in per_class.items()},
        index=classes,
    )
    table.insert(0, "scored cells", [per_class[MODELS[0]].get(name, {}).get("support", 0) for name in classes])
    return table.round(2)


def labeled_pairs(model, dimension):
    """Human label and model answer per image, for the scored cells of a dimension."""
    pairs = solution[["image_id", dimension]].merge(
        submissions[model][["image_id", dimension]], on="image_id", suffixes=("_label", "_answer")
    )
    pairs.columns = ["image_id", "label", "answer"]
    return pairs.dropna(subset=["label"])


def confusion(model, dimension):
    """Rows are the human label, columns the model's answer, in percent of each row."""
    pairs = labeled_pairs(model, dimension)
    table = pd.crosstab(pairs["label"], pairs["answer"].fillna("unreadable"), normalize="index")
    order = [*CLASSES[dimension], "unreadable"]
    table = table.reindex(index=[n for n in order if n in table.index], columns=[n for n in order if n in table.columns])
    return (100 * table).round().astype(int)


for dimension in DIMENSIONS:
    if dimension in metrics[MODELS[0]]:
        print(f"\n{dimension}: F1 per class")
        display(class_table(dimension))


subject: F1 per class


,scored cells,apertus-v1-5-8b,apertus-v1-5-70b
full_product,285,0.75,0.75
detail_closeup,34,0.00,0.00
accessories,4,0.00,0.00
packaging,15,0.00,0.00
package_and_product,15,0.00,0.00
multipack_trade_unit,4,0.00,0.00
delivery_scope,18,0.00,0.00
product_grouping,5,0.00,0.00
multiview,7,0.00,0.00
energy_label,22,0.00,0.00



background: F1 per class


,scored cells,apertus-v1-5-8b,apertus-v1-5-70b
white,354,0.86,0.93
color,33,0.13,0.07
scene_lifestyle,26,0.65,0.42



perspective: F1 per class


,scored cells,apertus-v1-5-8b,apertus-v1-5-70b
front_head_on,100,0.49,0.48
front_angle_left,49,0.00,0.00
front_angle_right,49,0.00,0.00
back_head_on,40,0.04,0.09
back_angle_left,17,0.00,0.00
back_angle_right,11,0.00,0.00
left_side,11,0.17,0.00
right_side,16,0.00,0.00
top_down,25,0.00,0.00
bottom_up,10,0.00,0.00


... (2 outputs left out)


text_classification: F1 per class


,scored cells,apertus-v1-5-8b,apertus-v1-5-70b
nutritional_info,25,0.0,0.76
technical_specs,25,0.0,0.00
promotional_text,15,0.0,0.00
certification_seal,1,0.0,0.00
other,10,0.0,0.00
none,383,0.9,0.87



text_language: F1 per class


,scored cells,apertus-v1-5-8b,apertus-v1-5-70b
German,52,0.73,0.59
English,67,0.54,0.19
French,6,0.44,0.00
Italian,1,0.00,0.00
Dutch,1,0.00,0.00
other,12,0.00,0.15
none,71,0.75,0.61


In [11]:
for model in MODELS:
    print(f"\nperspective, {model}: rows human label, columns answer, percent of each row")
    display(confusion(model, "perspective"))


perspective, apertus-v1-5-8b: rows human label, columns answer, percent of each row


answer,front_head_on,front_angle_right,back_head_on,back_angle_right,left_side,top_down
label,,,,,,
front_head_on,94,1,2,0,2,1
front_angle_left,94,0,0,0,6,0
front_angle_right,100,0,0,0,0,0
back_head_on,78,0,2,10,10,0
back_angle_left,71,6,6,0,18,0
back_angle_right,82,0,0,0,18,0
left_side,73,0,0,0,27,0
right_side,75,0,0,0,25,0
top_down,80,0,8,0,12,0



perspective, apertus-v1-5-70b: rows human label, columns answer, percent of each row


answer,front_head_on,back_head_on,left_side,top_down
label,,,,
front_head_on,100,0,0,0
front_angle_left,100,0,0,0
front_angle_right,96,2,0,2
back_head_on,95,5,0,0
back_angle_left,100,0,0,0
back_angle_right,91,0,9,0
left_side,100,0,0,0
right_side,100,0,0,0
top_down,96,4,0,0
